# Lab 4-2: Unsupervised learning.

In this notebook, we will explore some unsupervised learning techniques. We will visualize data by projecting it onto a low-dimensional space with the use of **Principal Component Analysis (PCA)**, **t-Distributed Stochastic Neighbor Embedding (t-SNE)** and **Uniform Manifold Approximation and Projection (UMAP)**. We will also cluster data using the **K-means** algorithm in order to predict the number of classes in a dataset.

***

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/mnist_train.csv')
df.describe()

train_X = df.drop('label', axis=1).to_numpy()
train_y = df['label'].to_numpy()

train_X = train_X / 255.0 # normalize the data, as pixel values are between 0 and 255

In [ ]:
train_X.shape

Take a look at the shape of our dataset as an array. The shape of the train_X is $(60000, 784)$, which basically means that there are **60000 images**, each of which is represented by a **vector of size $(784)$**, corresponding to all 784 pixels in an image, but flattened into a single vector, row by row. We know that the images in mnist dataset are $28 \times 28$ pixels, so the total number of pixels is $28 \times 28 = 784$.

## Reshaping arrays

**In order to display a single image from the dataset, we have to first convert it to a 2D table of numbers**, as raster graphics are just this (a table of pixel brightness values). We can do this by reshaping the vector of size $(784)$ to a matrix of size $(28, 28)$, basically folding the vector into a square.

After reshaping, the shape of the array should be $(60000, 28, 28)$, which means that there are **60000 images**, each of which is a **matrix of size $(28, 28)$**.

<center>
<img src="imgs/reshaping.png" width=600>
</center>

We can use the `reshape` method of a numpy array to reshape it. The method takes the new dimensions as arguments.

If we have an array of shape $(4, 4)$, we can reshape it to $(4, 2, 2)$ by calling `reshape(4, 2, 2)`. The number of elements in the original array must be equal to the number of elements in the reshaped array, otherwise an error will be raised.

If we do not know or care about the size of one of the dimensions, we can pass `-1` as the value of that dimension, and numpy will infer it from the other dimensions. For example, if we have an array of shape $(4, 4)$, we can call `reshape(-1, 2, 2)` to reshape it to $(4, 2, 2)$. The first dimension is inferred to be $4$ because $4 \times 2 \times 2 = 16$, which is the number of elements in the original array.
## Tensors

Both the original and reshaped arrays are examples of tensors. **A tensor is a generalization of a vector or matrix to any dimension**. In the figure above, we see how reshaping a tensor of shape $(4, 4)$ produces a tensor of shape $(4, 2, 2)$. 

The rank of a tensor is the number of dimensions it has. For example, the tensor of shape $(4, 2, 2)$ has rank 3, a tensor of shape $(4, 4)$, which is a **matrix** has rank 2, and a tensor of shape $(4)$, which is a **vector**, has rank 1.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_X_reshaped = train_X.reshape(-1, 28, 28) # reshape to 28x28, which is the size of a single image

i = 0 # see the i-th image

# seaborn heatmap is suitable for displaying simple raster graphics
sns.heatmap(train_X_reshaped[i], cmap='gray', xticklabels=False, yticklabels=False)
print(f"This is a {train_y[i]}:")

## Principal Component Analysis (PCA)

Principal Component Analysis (PCA) is a simple in concept but powerful unsupervised method for dimensionality reduction in data. It works by finding the directions in which the data varies the most, and then projects the data onto these directions.

PCA is a linear method, which means that it can only capture linear relationships in the data. However, it is a very useful method for visualizing high-dimensional data in 2D. Each data point can be represented as a linear combination of the principal components, which are the directions in which the data varies the most.

In [ ]:
# PCA
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler

pca = PCA(n_components=36)
pca.fit(train_X) # fit the PCA model to the data
train_X_pca = pca.transform(train_X) # transform the data to the principal components
train_X_pca

In [ ]:
# plot the first two components for some data

sns.set_style('white')
sns.set_context('notebook')
plt.figure(figsize=(10, 8))

# plot the first two principal component
plot_df = pd.DataFrame(train_X_pca[:,:2], columns=['PC1', 'PC2'])
plot_df['label'] = train_y

# You would normally use sns.scatterplot here, but I wanted to display each data point as a digit.
# For some reason, sns.scatterplot does not support muliple markers, but sns.lmplot does.
# When plotting PCA for any other data, I suggest using sns.scatterplot instead.

markers = [f"${i}$" for i in range(10)]
sns.lmplot(data=plot_df[:1000], x='PC1', y='PC2', hue='label', fit_reg=False, legend=False, markers=markers, palette='colorblind', height=6, aspect=1.5)
sns.despine(right=False, top=False)

In [ ]:
# show the first 4 principal components side by side

first_components = pca.components_[:4] # take the first 4 components

plt.figure(figsize=(20, 4))
for i, component in enumerate(first_components):
    plt.subplot(1, 4, i+1)
    sns.heatmap(component.reshape(28, 28), cmap='gray', xticklabels=False, yticklabels=False, robust=True)
    plt.title(f"Component {i+1}")

## Exercise 1: Reconstruct images from principal components (1 point)

We know that PCA decomposes the data into principal components, which are the directions in which the data varies the most. Each data point can be represented as a linear combination of these components:

$$ x = \sum_{i=1}^{n} \alpha_i v_i $$

where $x$ is the original data point, $n$ is the number of components, $\alpha_i$ are the coefficients of the linear combination, and $v_i$ are the principal components.

<center>
<img src="imgs/pca-combination.png" width=600>
</center>

We can extract the principal components from the PCA object using the `components_` attribute.

1. Extract the first 36 principal components from the PCA object.
2. Take a single data point from the dataset, and transform it to the principal components using the `transform` method of the PCA object.
3. Reconstruct the data point from the principal components using the formula above.
4. Plot the original data point and the reconstructed data point side by side using `sns.heatmap`.

 

## *How many components to keep?

The number of components to keep is a hyperparameter of the PCA algorithm, and it is usually chosen based on the amount of variance we want to capture in the data. The more components we keep, the more variance we capture, but the more complex the model becomes.

We can use the `explained_variance_ratio_` attribute of the PCA object to see how much variance is captured by each component. The attribute is an array of size equal to the number of components, and each element represents the proportion of variance captured by that component.

* How many components do we need to capture 90% of the variance in the data?

In [ ]:
ratio = pca.explained_variance_ratio_

sns.set_context('talk')
sns.set_style('whitegrid')

sns.lineplot(x=range(1, len(ratio)+1), y=np.cumsum(ratio))
plt.xticks(range(1, len(ratio)+1, 5))
plt.xlabel('Number of components')
plt.ylabel('Cumulative expl. variance')

## t-Distributed Stochastic Neighbor Embedding (t-SNE)

From a practical standpoint, these are the three main differences between PCA and t-SNE:

1. As data visualization techniques, **PCA is better at preserving the global structure**, or overall shape of the data, while **t-SNE is better at preserving the local relationships between data points**.
2. PCA can be fit to some data and then applied to transform new data, while t-SNE must be fit to all the data at once. **You can't embed new data points into an existing t-SNE projection without re-fitting the entire model.**
3. **t-SNE is computationally expensive**, and can take a long time to run on large datasets. PCA is much faster, and can be applied to large datasets with ease.

In [ ]:
# TSNE
from sklearn.manifold import TSNE

train_X_sample = train_X[:10000] # take a subset of the data to speed up the computation

# you can modify the perplexity and max_iter parameters to see how it affects the result
tsne = TSNE(n_components=2, random_state=42, max_iter=500, perplexity=30)
train_X_tsne = tsne.fit_transform(train_X_sample)
train_X_tsne

In [ ]:
# plot the result

sns.set_style('white')
sns.set_context('notebook')
plt.figure(figsize=(10, 8))

plot_df = pd.DataFrame(train_X_tsne, columns=['DIM1', 'DIM2'])
plot_df['label'] = train_y[:10000]

sns.lmplot(data=plot_df.sample(1000), x='DIM1', y='DIM2', hue='label', fit_reg=False, legend=False, markers=markers, palette='colorblind', height=6, aspect=1.5)
sns.despine(right=False, top=False)

## Uniform Manifold Approximation and Projection (UMAP)

UMAP is a newer dimensionality reduction technique that is similar to t-SNE, but is faster and can be applied to larger datasets. It also allows for more control over the embedding, and can be used to embed new data points into an existing projection.

UMAP is not implemented in scikit-learn, but there is a package called `umap-learn` that provides a similar interface to scikit-learn, and can be used in the same way.

In [ ]:
import umap

train_X_sample = train_X[:10000] # take a subset of the data to speed up the computation

umap_model = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1)
train_X_umap = umap_model.fit_transform(train_X_sample)

plot_df = pd.DataFrame(train_X_umap, columns=['DIM1', 'DIM2'])
plot_df['label'] = train_y[:10000]

sns.lmplot(data=plot_df[:500], x='DIM1', y='DIM2', hue='label', fit_reg=False, legend=False, markers=markers, palette='colorblind', height=6, aspect=1.5)
sns.despine(right=False, top=False)

## K-means clustering

Clustering is an unsupervised learning technique that groups data points into clusters based on their similarity. The goal of clustering is to find groups of data points that are similar to each other, and different from data points in other clusters.

The simplest clustering algorithm is K-means, which works by finding the centers of the clusters, and then assigning each data point to the cluster with the closest center. The number of clusters is a hyperparameter of the algorithm, and must be chosen by the user.

K-means is an iterative algorithm that works as follows:

1. Initialize the cluster centers randomly.

$$ c_i = \text{random}(\text{min}(x_i), \text{max}(x_i)) $$

2. Assign each data point to the cluster, whose center is closest to it. By closest, we usually mean the Euclidean distance between the data point and the cluster center.

$$ d(x, c) = \sqrt{\sum_{i=1}^{n} (x_i - c_i)^2} $$

3. Update the cluster centers by taking the mean of all the data points assigned to the cluster.

$$ c_i = \frac{1}{n} \sum_{j=1}^{n} x_{ij} $$

4. Repeat steps 2 and 3 until the cluster centers do not change significantly, or a maximum number of iterations is reached.

During the optimization process, K-means tries to minimize the sum of squared distances between the data points and their cluster centers. This is called the inertia of the clustering, and can be accessed using the `inertia_` attribute of the KMeans object.

## Exercise 2: Clustering beans (2 points)

<center>
<img src="imgs/bean.png" width=300>
</center>

Dry beans dataset contains various measurements and descriptors of different species of dried beans. As this dataset is publicly available, I removed some of the species present the dataset to make this exercise more interesting. The beans are unlabeled (the species is not known) and your task is to identify how many species are present in the dataset and **assign each data point to a cluster of beans of the same species.**

1. Load the dataset `data/dry-bean-unlabeled.csv` and normalize the data using `StandardScaler`.
2. Prepare PCA and UMAP projections of the dataset. How many different bean species do you think are present in the data?
3. Use K-means algorithm to cluster the data. You can use the `KMeans` class from scikit-learn.
4. Load the labels from the file `data/dry-bean-labels.csv` and compare the predicted clusters to the true labels. Which cluster corresponds to which species? How many data points of each class were assigned to a wrong cluster? Prepare some nice plots to visualize the results.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv('data/dry-bean-unlabeled.csv')
df.head()

# your code goes here
...

## *Elbow method

**The elbow method is a heuristic used to determine the optimal number of clusters in a dataset.** It works by plotting the inertia of the clustering as a function of the number of clusters, and looking for the "elbow" point in the plot. The elbow point is a certain number of clusters, after which the inertia starts decreasing linearly.

1. Fit the K-means algorithm to the dataset for different numbers of clusters, ranging from 1 to 10. For each fit, record the inertia of the clustering by accessing the `inertia_` attribute of the KMeans object.
2. Plot the inertia as a function of the number of clusters. Look for the elbow point in the plot. Does it match your intuition from the previous exercise?

In [ ]:
# your code goes here
...